In [16]:
import sys
import os

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.LoadupSamples import LoadupSamples
from src.hyperparameterTuning.HelperMetrics import HelperMetrics

import pandas as pd
import numpy as np
import polars as pl
import datetime
import random
import matplotlib.pyplot as plt
import logging
import time
import re

In [17]:
N = 5
T = 1
arr_open = [1.000,
            1.000,
            1.000,
            1.000,
            1.000]

arr_low  = [0.979,
            0.970,
            0.990,
            1.020,
            0.995]

arr_high = [1.005,
            1.005,
            1.025,
            1.040,
            1.008]

arr      = [1.000,  # fallback if no hit
            0.975,
            1.000,
            1.030,
            1.010]

arr = np.array(arr)
arr_low =  np.array(arr_low)
arr_high = np.array(arr_high)
arr_open = np.array(arr_open)

sl = 0.980
tp = 1.020
spread_cost = 0.000
commission = 0.000
out = HelperMetrics.collapse_sl_tp(
    arr, arr_low, arr_high, arr_open,
    sl=sl, tp=tp, spread_cost=spread_cost, commission=commission
)
out

array([0.98, 0.98, 1.02, 1.02, 1.01])

In [18]:
N = 5
T = 1
arr_open = [[1.000],
            [1.000],
            [1.000],
            [1.000],
            [1.000]]

arr_low  = [[0.979],
            [0.970],
            [0.990],
            [1.020],
            [0.995]]

arr_high = [[1.005],
            [1.021],
            [1.025],
            [1.040],
            [1.008]]

arr      = [[1.000],  # fallback if no hit
            [0.975],
            [1.000],
            [1.030],
            [1.010]]

arr = np.array(arr)
arr_low =  np.array(arr_low)
arr_high = np.array(arr_high)
arr_open = np.array(arr_open)

sl = 0.980
tp = 1.020
spread_cost = 0.000
commission = 0.000
out = HelperMetrics.collapse_sl_tp(
    arr, arr_low, arr_high, arr_open,
    sl=sl, tp=tp, spread_cost=spread_cost, commission=commission
)
# array([0.98, 0.98, 1.02, 1.02, 1.01])
out

array([0.98, 0.98, 1.02, 1.02, 1.01])

In [19]:
# Five rows demonstrate: SL-by-low, SL-by-open-gap, TP-by-high, TP-by-open-gap, No-hit
arr = np.array([
    [1.00, 1.00, 1.00],   # 0: SL by low (non-gap) at j=0 -> sl - spread_cost
    [1.00, 1.00, 1.00],   # 1: SL by open gap at j=0      -> open
    [1.00, 1.00, 1.00],   # 2: TP by high (non-gap) at j=0-> tp - spread_cost
    [1.00, 1.00, 1.00],   # 3: TP by open gap at j=0      -> open
    [1.00, 1.005, 1.01],  # 4: No hit                      -> last value
])
arr_open = np.array([
    [1.00, 1.00, 1.00],
    [1.00, 1.00, 1.00],
    [1.00, 1.00, 1.00],
    [1.00, 1.00, 1.00],
    [1.00, 1.005, 1.01],
])
arr_low = np.array([
    [0.97, 0.99, 0.99],
    [0.96, 0.99, 0.99],
    [0.99, 0.99, 0.99],
    [0.99, 0.99, 0.99],
    [0.985,0.99, 0.99],
])
arr_high = np.array([
    [1.01, 1.01, 1.01],
    [1.01, 1.01, 1.01],
    [1.03, 1.01, 1.01],
    [1.04, 1.01, 1.01],
    [1.015,1.015,1.015],
])
sl, tp = 0.98, 1.02
out = HelperMetrics.collapse_sl_tp(
    arr, arr_low, arr_high, arr_open,
    sl=sl, tp=tp, spread_cost=0.005, commission=0.0
)
out
#array([0.975, 0.975, 1.015, 1.015, 1.01 ])
# Row-wise interpretation:
# 0 -> 0.98 - 0.005 = 0.975 (SL by low, non-gap)
# 1 -> 0.97           (SL by open gap)
# 2 -> 1.02 - 0.005 = 1.015 (TP by high, non-gap)
# 3 -> 1.03           (TP by open gap)
# 4 -> 1.01           (no threshold hit -> last value)


array([0.975, 0.975, 1.015, 1.015, 1.01 ])

In [20]:
out = HelperMetrics.collapse_sl_tp(
    arr, arr_low, arr_high, arr_open,
    sl=sl, tp=None, spread_cost=0.005, commission=0.0
)
out
# array([0.975, 0.975, 1.   , 1.   , 1.01 ])

array([0.975, 0.975, 1.   , 1.   , 1.01 ])

### Test collapse_perstep

In [21]:
def run_case(name, arr, arr_low, arr_high, arr_open, sl, tp, spread_cost=0.0, commission=0.0):
    out = HelperMetrics.collapse_sl_tp(arr, arr_low, arr_high, arr_open, sl, tp, spread_cost, commission)
    print(f"{name}:")
    print("  out =", out)
    print()
    return out

In [22]:
# 1 path, 3 steps; SL/TP far away, no hit expected
arr      = np.array([[100.0, 101.0, 102.0]])  # close
arr_low  = np.array([[ 99.0, 100.0, 101.0]])
arr_high = np.array([[101.0, 102.0, 103.0]])
arr_open = np.array([[100.0, 101.0, 102.0]])

sl = np.full_like(arr,  90.0)
tp = np.full_like(arr, 110.0)

out = run_case("No hit (should exit at last close, 102)", 
               arr, arr_low, arr_high, arr_open, sl, tp)

assert np.allclose(out, np.array([102.0]))

No hit (should exit at last close, 102):
  out = [102.]



In [23]:
# First bar: open above SL, low below SL => SL non-gap
arr      = np.array([[100.0, 101.0]])
arr_low  = np.array([[ 94.0,  99.0]])
arr_high = np.array([[105.0, 103.0]])
arr_open = np.array([[100.0, 101.0]])

sl = np.array([[ 95.0,  95.0]])
tp = np.array([[110.0, 110.0]])

spread_cost = 0.1

out = run_case("SL non-gap on t=0 (expected 95 - 0.1 = 94.9)", 
               arr, arr_low, arr_high, arr_open, sl, tp, spread_cost)

assert np.allclose(out, np.array([95.0 - spread_cost]))

SL non-gap on t=0 (expected 95 - 0.1 = 94.9):
  out = [94.9]



In [24]:
# First bar: open <= SL => SL gap; fill at open, no spread
arr      = np.array([[100.0,  99.0]])
arr_low  = np.array([[ 94.0,  98.0]])
arr_high = np.array([[101.0, 100.0]])
arr_open = np.array([[ 94.0,  99.0]])  # big gap down

sl = np.array([[ 95.0,  95.0]])
tp = np.array([[110.0, 110.0]])

spread_cost = 0.1

out = run_case("SL gap on t=0 (expected open=94.0, no spread)", 
               arr, arr_low, arr_high, arr_open, sl, tp, spread_cost)

assert np.allclose(out, np.array([94.0]))

SL gap on t=0 (expected open=94.0, no spread):
  out = [94.]



In [25]:
# First bar: open below TP, high above TP => TP non-gap
arr      = np.array([[100.0, 101.0]])
arr_low  = np.array([[ 99.0, 100.0]])
arr_high = np.array([[106.0, 103.0]])
arr_open = np.array([[100.0, 101.0]])

sl = np.array([[ 90.0,  90.0]])
tp = np.array([[105.0, 105.0]])

spread_cost = 0.1

out = run_case("TP non-gap on t=0 (expected 105 - 0.1 = 104.9)", 
               arr, arr_low, arr_high, arr_open, sl, tp, spread_cost)

assert np.allclose(out, np.array([105.0 - spread_cost]))

TP non-gap on t=0 (expected 105 - 0.1 = 104.9):
  out = [104.9]



In [26]:
# First bar: open >= TP => TP gap; fill at open, no spread
arr      = np.array([[100.0, 101.0]])
arr_low  = np.array([[ 99.0, 100.0]])
arr_high = np.array([[108.0, 103.0]])
arr_open = np.array([[107.0, 101.0]])  # gap up

sl = np.array([[ 90.0,  90.0]])
tp = np.array([[105.0, 105.0]])

spread_cost = 0.1

out = run_case("TP gap on t=0 (expected open=107.0, no spread)", 
               arr, arr_low, arr_high, arr_open, sl, tp, spread_cost)

assert np.allclose(out, np.array([107.0]))

TP gap on t=0 (expected open=107.0, no spread):
  out = [107.]



In [27]:
# Bar hits both SL and TP; SL should win
arr      = np.array([[100.0, 101.0]])
arr_low  = np.array([[ 94.0, 100.0]])   # below SL
arr_high = np.array([[106.0, 103.0]])   # above TP
arr_open = np.array([[100.0, 101.0]])

sl = np.array([[ 95.0,  95.0]])
tp = np.array([[105.0, 105.0]])

spread_cost = 0.1

out = run_case("Both SL & TP non-gap on t=0 (SL priority, expected 95 - 0.1 = 94.9)", 
               arr, arr_low, arr_high, arr_open, sl, tp, spread_cost)

assert np.allclose(out, np.array([95.0 - spread_cost]))

Both SL & TP non-gap on t=0 (SL priority, expected 95 - 0.1 = 94.9):
  out = [94.9]



In [28]:
# No hit on t=0, TP hit on t=1
arr      = np.array([[100.0, 102.0, 104.0]])
arr_low  = np.array([[ 99.0, 101.0, 103.0]])
arr_high = np.array([[101.0, 106.0, 105.0]])
arr_open = np.array([[100.0, 101.0, 104.0]])

sl = np.array([[ 90.0,  90.0,  90.0]])
tp = np.array([[110.0, 105.0, 105.0]])

spread_cost = 0.1

out = run_case("Multi-step TP hit at t=1 (expected 105 - 0.1 = 104.9)", 
               arr, arr_low, arr_high, arr_open, sl, tp, spread_cost)

assert np.allclose(out, np.array([105.0 - spread_cost]))

Multi-step TP hit at t=1 (expected 105 - 0.1 = 104.9):
  out = [104.9]



In [29]:
# Simple: TP non-gap, plus commission
arr      = np.array([[100.0, 101.0]])
arr_low  = np.array([[ 99.0, 100.0]])
arr_high = np.array([[106.0, 103.0]])
arr_open = np.array([[100.0, 101.0]])

sl = np.array([[ 90.0,  90.0]])
tp = np.array([[105.0, 105.0]])

spread_cost = 0.0
commission  = 0.001  # 0.1% per side

raw_exit = 105.0
expected = raw_exit * (1 - commission) * (1 - commission)

out = run_case("TP non-gap + commission (expected 105 * (1-0.001)^2)", 
               arr, arr_low, arr_high, arr_open, sl, tp, spread_cost, commission)

assert np.allclose(out, np.array([expected]))

TP non-gap + commission (expected 105 * (1-0.001)^2):
  out = [104.790105]



In [30]:
# 3 paths:
# 0: SL non-gap
# 1: TP gap
# 2: no hit -> last close
arr      = np.array([
    [100.0, 101.0],  # path 0
    [100.0, 101.0],  # path 1
    [100.0, 102.0],  # path 2
])
arr_low  = np.array([
    [ 94.0,  99.0],  # hits SL
    [ 99.0,  99.0],
    [ 99.0, 101.0],
])
arr_high = np.array([
    [105.0, 103.0],
    [108.0, 107.0],  # high irrelevant, TP gap
    [101.0, 103.0],
])
arr_open = np.array([
    [100.0, 101.0],
    [107.0, 101.0],  # TP gap at t=0
    [100.0, 101.0],
])

sl = np.array([
    [ 95.0,  95.0],
    [ 90.0,  90.0],
    [ 90.0,  90.0],
])
tp = np.array([
    [110.0, 110.0],
    [105.0, 105.0],
    [110.0, 110.0],
])

spread_cost = 0.1
commission  = 0.0

out = run_case("Multiple paths", 
               arr, arr_low, arr_high, arr_open, sl, tp, spread_cost, commission)

expected = np.array([
    95.0 - spread_cost,  # SL non-gap
    107.0,               # TP gap at open
    102.0,               # no hit, last close
])

assert np.allclose(out, expected)

Multiple paths:
  out = [ 94.9 107.  102. ]

